# Chapter 5: Roofline Focused Notebook

This notebook isolates the roofline-style judgment in Chapter 5. It shows how to classify a kernel profile and decide the next optimization step.

## What this notebook teaches

- How to read arithmetic intensity
- How to classify a profile as memory-bound or compute-bound
- How to turn the classification into a concrete next step

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "HANDOFF.md").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.benchmark_record_template import set_benchmark_record
from examples.ch05_schedule_optimize_core import classify, get_profiles, next_step

peak_compute_tflops = 312.0
peak_memory_gbs = 2000.0

print('Chapter 5: Roofline Analysis')
print('=' * 72)
print(f'Peak compute: {peak_compute_tflops:.1f} TFLOPS')
print(f'Peak memory:  {peak_memory_gbs:.1f} GB/s')
print()

for profile in get_profiles():
    bottleneck = classify(profile, peak_compute_tflops, peak_memory_gbs)
    print(f'{profile.name}')
    print(f'  FLOPs: {profile.flops:.3e}')
    print(f'  Bytes: {profile.bytes_accessed:.3e}')
    print(f'  AI:    {profile.arithmetic_intensity:.2f} FLOP/Byte')
    print(f'  Class: {bottleneck}')
    print(f'  Next:  {next_step(profile, peak_compute_tflops, peak_memory_gbs)}')
    print()

record = set_benchmark_record(
    scenario='chapter 5 roofline analysis',
    operator='analysis',
    platform='NVIDIA',
    target='cuda',
    dtype='float32',
    shape='analysis only',
    baseline='roofline estimate',
    optimization='shared memory / pipeline / layout',
)
print('Benchmark record skeleton:')
for key in ['scenario', 'operator', 'platform', 'target', 'dtype', 'shape', 'baseline', 'optimization']:
    print(f'  {key}: {record[key]}')


## Takeaway

Roofline is a judgment tool, not the result itself. It tells you whether to push reuse or compute efficiency next.